#Load Everything (Model + Tools)

In [10]:
import joblib
import requests
import io
import pandas as pd
import ee


In [11]:



# Base URL to the raw files
url_base = "https://raw.githubusercontent.com/irungus/tree-species-ml-pipeline/main/models"

def load_from_url(file_name):
    full_url = f"{url_base}/{file_name}"
    response = requests.get(full_url)

    # Check if the request was successful
    if response.status_code == 200:
        return joblib.load(io.BytesIO(response.content))
    else:
        raise Exception(f"Failed to download {file_name}. Status code: {response.status_code}")

# Load your components
model = load_from_url("best_model.pkl")
scaler = load_from_url("scaler.pkl")
le = load_from_url("label_encoder.pkl")
feature_names = load_from_url("feature_names.pkl")

print("All components loaded successfully!")

All components loaded successfully!


In [12]:

ee.Authenticate()
ee.Initialize(project='agfkenya')

In [13]:
def extract_features(lat, lon):
    point = ee.Geometry.Point([lon, lat])

    # Sentinel-2
    s2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .filterDate("2022-01-01", "2022-12-31")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        .select(["B2", "B4", "B8"])
        .median()
    )

    ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")

    evi = s2.expression(
        "2.5 * ((NIR - RED) / (NIR + 6*RED - 7.5*BLUE + 1))",
        {
            "NIR": s2.select("B8"),
            "RED": s2.select("B4"),
            "BLUE": s2.select("B2"),
        },
    ).rename("EVI")

    savi = s2.expression(
        "((NIR - RED) / (NIR + RED + 0.5)) * 1.5",
        {
            "NIR": s2.select("B8"),
            "RED": s2.select("B4"),
        },
    ).rename("SAVI")

    # SAR
    s1 = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(point)
        .filterDate("2022-01-01", "2022-12-31")
        .select(["VV", "VH"])
        .median()
    )

    vv = s1.select("VV")
    vh = s1.select("VH")
    ratio = vv.divide(vh).rename("VV_VH_ratio")

    # Terrain
    dem = ee.Image("USGS/SRTMGL1_003")
    slope = ee.Terrain.slope(dem)

    # Climate
    climate = (
        ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY")
        .filterBounds(point)
        .filterDate("2022-01-01", "2022-12-31")
    )

    temp = climate.select("temperature_2m").mean()
    precip = climate.select("total_precipitation").sum()

    # Combine
    image = ee.Image.cat([
        ndvi, evi, savi,
        vv.rename("VV"), vh.rename("VH"), ratio,
        dem.rename("elevation"), slope.rename("slope"),
        temp.rename("temperature_2m"),
        precip.rename("total_precipitation")
    ])

    values = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=point,
        scale=30,
        bestEffort=True
    )

    return values.getInfo()

In [14]:
def prepare_input(features_dict):
    df = pd.DataFrame([features_dict])

    # Ensure correct columns
    df = df.reindex(columns=feature_names)

    # Handle missing values (important!)
    df = df.fillna(df.mean())

    # Scale
    X_scaled = scaler.transform(df)

    return X_scaled

In [15]:
def predict_species(lat, lon):
    features = extract_features(lat, lon)

    if not features:
        return "No data available"

    X = prepare_input(features)

    pred = model.predict(X)

    return le.inverse_transform(pred)[0]

In [18]:
lat = -0.7166
lon = 36.0800  # Nakuru area

prediction = predict_species(lat, lon)

print("Predicted species:", prediction)

Predicted species: Grevillea


In [19]:
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/predict", methods=["GET"])
def predict():
    lat = float(request.args.get("lat"))
    lon = float(request.args.get("lon"))

    result = predict_species(lat, lon)

    return jsonify({"species": result})

app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
